# CHARTA Step 34 — Fine-tune ClinicalBERT with LoRA on BC5CDR (CID Relation Extraction)

This notebook fine-tunes `emilyalsentzer/Bio_ClinicalBERT` for Chemical-Induced Disease (CID)
relation extraction using LoRA on the BC5CDR dataset. Designed for **Google Colab T4 GPU**.

**Workflow:**
1. Install dependencies
2. Clone CHARTA repo & set up path
3. Run fine-tuning
4. Download LoRA weights to local machine
5. (Optional) Evaluate NER F1

After downloading, place weights in `CHARTA/models/lora_weights/clinicalbert_rel/` locally.

In [ ]:
#@title Step 1 — Install dependencies  {display-mode: "form"}
!pip install -q transformers datasets peft accelerate scispacy scikit-learn
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
print("✅ Dependencies installed")

In [ ]:
#@title Step 2 — Clone CHARTA & set PYTHONPATH  {display-mode: "form"}
import os, sys

# Option A: Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/<YOUR_USERNAME>/CHARTA.git /content/CHARTA

# Option B: Upload CHARTA folder directly to Colab
#   - In the Colab file browser, click Upload → select your local CHARTA/ folder
#   - It will appear at /content/CHARTA

CHARTA_ROOT = "/content/CHARTA"
assert os.path.isdir(CHARTA_ROOT), f"CHARTA not found at {CHARTA_ROOT}. Upload or clone first."
sys.path.insert(0, os.path.join(CHARTA_ROOT, "src"))
os.chdir(CHARTA_ROOT)
print(f"✅ PYTHONPATH set, working dir: {os.getcwd()}")

In [ ]:
#@title Step 3 — Verify GPU  {display-mode: "form"}
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — switch Colab runtime to T4 GPU")

In [ ]:
#@title Step 4 — Fine-tune ClinicalBERT with LoRA  {display-mode: "form"}
from layer2.trainer import finetune_relation_model

# This will:
#   1. Load BC5CDR (bigbio/bc5cdr, bc5cdr_bigbio_kb config)
#   2. Build entity-pair classification rows
#   3. Apply LoRA (r=8, alpha=16, target=[query,value])
#   4. Train for 10 epochs with fp16 on T4
#   5. Save LoRA weights to models/lora_weights/clinicalbert_rel/

finetune_relation_model(
    dataset_name="bigbio/bc5cdr",
    config_name="bc5cdr_bigbio_kb",
    output_dir="models/lora_weights/clinicalbert_rel/",
    num_epochs=10,
)

In [ ]:
#@title Step 5 — Verify saved LoRA weights  {display-mode: "form"}
import os

weights_dir = "models/lora_weights/clinicalbert_rel/"
expected_files = ["adapter_config.json", "adapter_model.safetensors"]
for f in expected_files:
    path = os.path.join(weights_dir, f)
    assert os.path.isfile(path), f"Missing: {path}"
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✅ {f} ({size_kb:.1f} KB)")

# Also check tokenizer files
tok_files = ["tokenizer.json", "tokenizer_config.json", "vocab.txt"]
for f in tok_files:
    path = os.path.join(weights_dir, f)
    if os.path.isfile(path):
        print(f"  ✅ {f} ({os.path.getsize(path)/1024:.1f} KB)")

print("\n✅ All LoRA weight files saved successfully")

In [ ]:
#@title Step 6 — Quick inference test on Colab  {display-mode: "form"}
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from shared.constants import CLINICALBERT_MODEL
import torch

# Load base model + LoRA adapter
tokenizer = AutoTokenizer.from_pretrained("models/lora_weights/clinicalbert_rel/")
base_model = AutoModelForSequenceClassification.from_pretrained(CLINICALBERT_MODEL, num_labels=2)
model = PeftModel.from_pretrained(base_model, "models/lora_weights/clinicalbert_rel/")
model.eval()

# Test with a sample CID sentence
test_input = "[E1] aspirin [/E1] The patient was treated with aspirin for [E2] myocardial infarction [/E2]"
inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=256)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    label = torch.argmax(logits, dim=-1).item()

print(f"Input   : {test_input}")
print(f"Label   : {label}  (0=No relation, 1=CID)")
print(f"Confidence: {probs[0][label].item():.4f}")
print("✅ Inference test passed")

In [ ]:
#@title Step 7 — (Optional) Evaluate NER F1  {display-mode: "form"}
from layer2.evaluator import evaluate_ner

# This evaluates scispaCy NER on NCBI Disease test split
# Expected: micro-F1 > 0.75
results = evaluate_ner()
print(f"NER F1: {results['ner_f1']:.4f}")

## Step 8 — Download LoRA weights to your local machine

Run the cell below to download the LoRA weights as a zip file. Then extract them into
`CHARTA/models/lora_weights/clinicalbert_rel/` on your local machine.

**Local folder structure after extraction:**
```
CHARTA/
├── models/
│   └── lora_weights/
│       └── clinicalbert_rel/
│           ├── adapter_config.json
│           ├── adapter_model.safetensors
│           ├── tokenizer.json
│           ├── tokenizer_config.json
│           └── vocab.txt
│           └── special_tokens_map.json
```

In [ ]:
#@title Step 8 — Download LoRA weights as zip  {display-mode: "form"}
import shutil
from google.colab import files

zip_path = shutil.make_archive("clinicalbert_rel_lora", "zip",
                              root_dir="models/lora_weights",
                              base_dir="clinicalbert_rel")
print(f"Zip created: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)")
files.download(zip_path)
print("✅ Download started — extract into CHARTA/models/lora_weights/clinicalbert_rel/ locally")

## Post-Download: Local Setup Instructions

After downloading and extracting the zip into `CHARTA/models/lora_weights/clinicalbert_rel/`:

1. **Verify locally:**
   ```bash
   cd CHARTA/src
   python -c "from layer2.relation_extractor import load_relation_model; t, m = load_relation_model(); print('LoRA model loaded OK')"
   ```

2. **Run Layer 2 pipeline with relations:**
   ```bash
   python run_layer2.py --mode run
   ```

3. **The relation extractor will now return CID relations instead of empty lists.**